In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/image/images.zip -d /content/images.zip
!unzip -q images.zip

In [ ]:
!rm /content/images.zip

In [ ]:
import os
A=os.listdir('/content')
A

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
s='/content/images/*'
path=glob.glob(s)
import re
def sorted_alphanumeric(data):
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [ convert(c) for c in re.split('([0-9]+)', key) ]
    return sorted(data, key=alphanum_key)

path1=sorted_alphanumeric(path)
path1 = [item.replace('p','_') for item in path1]
data=pd.read_excel('/content/drive/MyDrive/image/data_c.xlsx')
data
len(path1)
print(path1)
type(path1)
path1=np.array(path1)
path1.shape
path1[0]

In [ ]:
path1=np.array(path1)
data=np.array(data)

In [ ]:
df=pd.DataFrame({'imgpath':path1,'Porosity':data[:,0],'throat radius':data[:,1],'pore radius':data[:,2],'pore_connection_number':data[:,3],'pore shape factor':data[:,4]})

In [ ]:
df_new=df[df['Porosity'] <0.35 ]
df_new.shape
df_new

In [ ]:
df_new_five=df_new.iloc[::1,:]
df_new_five

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data1=np.array(df_new_five)
scaler = StandardScaler()
data1= scaler.fit_transform(data1[:,1:])
print(data1)
print(type(data1))
data1.shape

In [ ]:
df=pd.DataFrame({'imgpath':df_new_five['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

In [ ]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.25,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

In [ ]:
def Data_predict_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img_2 = []

                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    x_batch_img_2.append(img_1)



                x_batch_img_2 = np.array(x_batch_img_2)

              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img_2

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
def Data_generator(df, batch_size, augment=False):
  for start in range(0, df.shape[0], batch_size):
      x_batch_img = []
      y_batch=[]
      end = min(start + batch_size, df.shape[0])

      for idd in range(start,end):
          img = open(np.array(df.imgpath)[idd],'rb').read()
          img_1=np.frombuffer(img,dtype=np.uint8)
          img_1=img_1.reshape(100,100,100,1)
          Porosity=np.array(df.Porosity)[idd]
          throat_radius=np.array(df['throat radius'])[idd]
          pore_radius=np.array(df['pore radius'])[idd]
          pore_connection_number=np.array(df.pore_connection_number)[idd]
          pore_shape_factor=np.array(df['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          x_batch_img.append(img_1)
          y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      print(x_batch_img.shape)
      # x_batch_img = tf.squeeze(np.array(x_batch_img), axis=-1)
      y_batch= np.array(y_batch)
      y_batch=y_batch.reshape(-1,5)

      datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.2,height_shift_range=0.2,horizontal_flip=True)
      # datagen.fit(x_batch_img)
      data = datagen.flow(x_batch_img, y_batch, batch_size=32, shuffle=True)
      X = []
      while True:
        try:
          X.append(data.next())
        except:
          break
      # print(type(data))
    #y_batch = to_categorical(y_batch,5)
      yield np.array(X)

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import cv2
def apply_augmentation(image):
    final_augmented = []
    final_augmented.append(image)
    for i in range(3):
        rotation_angle = (i+1) * 90
        rotation_matrix = cv2.getRotationMatrix2D((100 // 2, 100 // 2), rotation_angle, 1.0)
        rotated_slices = [cv2.warpAffine(slice_2d, rotation_matrix, (100, 100)) for slice_2d in image]
        augmented_image = np.stack(rotated_slices, axis=0)
        final_augmented.append(augmented_image)
    for j in range(2):
        flipped_slices = [cv2.flip(slice_2d, j) for slice_2d in image]
        augmented_image_2 = np.stack(flipped_slices, axis=0)
        final_augmented.append(augmented_image_2)
    for k in range(2):
        rotation_matrix90 = cv2.getRotationMatrix2D((100 // 2, 100 // 2), 90, 1.0)
        rotated_slices90 = [cv2.warpAffine(slice_2d, rotation_matrix90, (100, 100)) for slice_2d in image]
        augmented_image90 = np.stack(rotated_slices90, axis=0)
        flipped_rotate_slices = [cv2.flip(slice_2d, k) for slice_2d in augmented_image90]
        augmented_image90_f =  np.stack(flipped_rotate_slices, axis=0)
        final_augmented.append(augmented_image90_f)

    return final_augmented


In [ ]:
x = np.zeros((5,5,5,1))
x2 = apply_augmentation(x)
len(x2)

In [ ]:
#این دیتا جنریتور است

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
def Data_generator(df, batch_size, augment=False):
  for start in range(0, df.shape[0], batch_size):
      x_batch_img = []
      y_batch = []
      end = min(start + batch_size, df.shape[0])

      for idd in range(start,end):
          img = open(np.array(df.imgpath)[idd],'rb').read()
          img_1 = np.frombuffer(img,dtype=np.uint8)
          #img_1 = img_1.reshape(100,100,100,1)
          img_1 = img_1.reshape(100,100,100)
          img_1 = apply_augmentation(img_1)
          img_1 = [each_image.reshape(100,100,100,1) for each_image in img_1]
          x_batch_img.extend(img_1)
          Porosity=np.array(df.Porosity)[idd]
          throat_radius=np.array(df['throat radius'])[idd]
          pore_radius=np.array(df['pore radius'])[idd]
          pore_connection_number=np.array(df.pore_connection_number)[idd]
          pore_shape_factor=np.array(df['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          for i in range(8):
            y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      y_batch = np.array(y_batch)
      y_batch = y_batch.reshape(-1,5)




      yield x_batch_img, y_batch

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
class Data_generator(tf.keras.utils.Sequence):
  def __init__(self, data, batch_size=4, dim=(100,100,100), channels=1, shuffle=False, augment=False):
    self.data = data
    self.batch_size=batch_size
    self.dim=dim
    self.channels=channels
    self.shuffle=shuffle
    self.on_epoch_end()

  def __len__(self):
    return int(np.floor(len(self.data) / self.batch_size))

  def __getitem__(self, index):
    indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
    # Generate data
    X, y = self.__data_generation(indexes)
    return X, y

  def on_epoch_end(self):

    'Updates indexes after each epoch'
    self.indexes = np.arange(len(self.data))
    if self.shuffle == True:
        np.random.shuffle(self.indexes)

  def __data_generation(self, indexes):
      x_batch_img = []
      y_batch = []

      for idd in indexes:
          img = open(np.array(self.data.imgpath)[idd],'rb').read()
          img_1 = np.frombuffer(img,dtype=np.uint8)
          #img_1 = img_1.reshape(100,100,100,1)
          img_1 = img_1.reshape(100,100,100)
          img_1 = apply_augmentation(img_1)
          img_1 = [each_image.reshape(100,100,100,1) for each_image in img_1]
          x_batch_img.extend(img_1)
          Porosity=np.array(self.data.Porosity)[idd]
          throat_radius=np.array(self.data['throat radius'])[idd]
          pore_radius=np.array(self.data['pore radius'])[idd]
          pore_connection_number=np.array(self.data.pore_connection_number)[idd]
          pore_shape_factor=np.array(self.data['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          for i in range(8):
            y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      y_batch = np.array(y_batch)
      y_batch = y_batch.reshape(-1,5)

      return x_batch_img, y_batch

In [ ]:
df_train.shape

In [ ]:
batch_size=4
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
train_gen_pre=Data_predict_generator(df_train,batch_size)
valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import math
ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
nbatches_train=math.ceil(df_train.shape[0]/batch_size)
nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
print(nbatches_valid,nbatches_train,nbatches_test)

In [ ]:
from keras.models import Sequential
from keras.layers import Conv3D,MaxPooling3D,Dropout,Flatten,Dense,BatchNormalization,ReLU
from tensorflow.keras import initializers
from keras.callbacks import *
import tensorflow as tf

In [ ]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(7,7,7),input_shape=(100,100,100,1),padding="same",activation='relu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(5,5,5),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

In [ ]:
import os
checkpoint_path="/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-{epoch:03d}.ckpt"
#os.makedirs("/content/drive/MyDrive/image/porosity_filtering/training_5layers", exist_ok=True)
#ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/image/porosity_filtering/training_model_5layers/weights.{epoch:02d}-{val_loss:.2f}.hdf5', monitor='val_loss')
cp_callback=ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1)

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/Training_199.log')

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-006.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen, callbacks=[cp_callback,csvlogger],initial_epoch=6)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-011.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen, callbacks=[cp_callback,csvlogger],initial_epoch=11)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-015.ckpt'
model.load_weights(wieght)

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen, callbacks=[cp_callback,csvlogger],initial_epoch=15)

In [ ]:
def debug_fit_generator(train_gen, model):
    for i, (x_batch_img, y_batch) in enumerate(train_gen):
        print("Batch:", i)
        print("x_batch_img shape (before model):", x_batch_img.shape)
        print("y_batch shape (before model):", y_batch.shape)

        # Model training step
        model.train_on_batch(x_batch_img, y_batch)

        # Print shapes after passing through the model
        x_batch_img_after_model = model.predict(x_batch_img)
        print("x_batch_img shape (after model):", x_batch_img_after_model.shape)
        print("y_batch shape (after model):", y_batch.shape)

        if i >= steps_per_epoch:
            break

debug_fit_generator(train_gen, model)


In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-021.ckpt'
model.load_weights(wieght)

In [ ]:
history1=model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen,callbacks=[cp_callback,csvlogger],initial_epoch=21)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-022.ckpt'
model.load_weights(wieght)

In [ ]:
history1=model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen,callbacks=[cp_callback,csvlogger],initial_epoch=22)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-023.ckpt'
model.load_weights(wieght)

In [ ]:
history1=model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen,callbacks=[cp_callback,csvlogger],initial_epoch=23)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-026.ckpt'
model.load_weights(wieght)

In [ ]:
history1=model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen, callbacks=[cp_callback,csvlogger],initial_epoch=26)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-028.ckpt'
model.load_weights(wieght)

In [ ]:
history1=model.fit_generator(train_gen,  epochs=200, verbose=1, validation_data=valid_gen,callbacks=[cp_callback,csvlogger],initial_epoch=28)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-040.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=40)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-042.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=42)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-043.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=43)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-049.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=49)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-058.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=58)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-059.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=59)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-065.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=65)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-067.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=67)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-073.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=73)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-075.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=76)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-091.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=91)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-092.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=92)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-112.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=112)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-118.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=118)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-124.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=124)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-132.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=132)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-133.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=133)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-134.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=134)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/shared/training_model_weights/cp-147.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=147)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-149.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=149)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-151.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=151)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-153.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=153)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-155.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=155)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-161.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=161)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-162.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=162)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-164.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=164)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-166.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=166)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-171.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=171)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-172.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=172)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-174.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=174)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-180.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=180)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-184.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=18)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-185.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=185)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-187.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=187)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-189.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=189)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-194.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=194)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-195.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=195)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-198.ckpt'
model.load_weights(wieght)

In [ ]:
history1 = model.fit_generator(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[cp_callback,csvlogger],initial_epoch=198)

In [ ]:
#**********************************************predicting***********************************************

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/data_aug/all_data/training_model_weights/cp-200.ckpt'
model.load_weights(wieght)

In [ ]:
model.evaluate_generator(test_gen, nbatches_test, workers=1)

In [ ]:
y=model.predict_generator(
    test_gen_pre,
    verbose=1,
    steps=nbatches_test,
    callbacks=None,
    max_queue_size=10,
    workers=1,
    use_multiprocessing=False)

In [ ]:
y_1=pd.DataFrame({'porosity':((y[:,0]*df_new['Porosity'].std())+df_new['Porosity'].mean()),
                  'throat radius':((y[:,1]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((y[:,2]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((y[:,3]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((y[:,4]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
df_test_1=np.array(df_test)
df_test_2=pd.DataFrame({'porosity':((df_test_1[:,1]*df_new.Porosity.std())+df_new.Porosity.mean()),
                  'throat radius':((df_test_1[:,2]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((df_test_1[:,3]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((df_test_1[:,4]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((df_test_1[:,5]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['porosity'])
y_values = np.array(y_1['porosity'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore shape factor'])
y_values = np.array(y_1['pore shape factor'])
r_squared_pore_shape_factor=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_shape_factor)

In [ ]:
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['throat radius'])
y_values = np.array(y_1['throat radius'])
r_squared_throat_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_throat_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore radius'])
y_values = np.array(y_1['pore radius'])
r_squared_pore_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore_connection_number'])
y_values = np.array(y_1['pore_connection_number'])
r_squared_pore_connection_number=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_connection_number)

In [ ]:
#تمرین_data_aug

In [ ]:
from keras.datasets import mnist

In [ ]:
(train_x,tarin_y),(test_x,test_y) = mnist.load_data()

In [ ]:
train_x

In [ ]:
type(train_x)

In [ ]:
train_x.shape

In [ ]:
print(f'x_train {train_x.shape}',f'y_tarin {tarin_y.shape}')

In [ ]:
from matplotlib import pyplot
for i in range(9):
  pyplot.subplot(330+1+i)
  pyplot.imshow(train_x[i],cmap='gray', vmin=0, vmax=255)
pyplot.show()

In [ ]:
import numpy as np
x = np.linspace(0, 2*np.pi, 400)
y = np.sin(x**2)
fig, ax=pyplot.subplots(3,3,sharex=True,sharey=True)
ax[0][0].plot(x,y)

In [ ]:
fig,[[ax1,ax2],[ax3,ax4]]=pyplot.subplots(2,2)
ax3.plot(x,y)

In [ ]:
ax

In [ ]:
type(x)

In [ ]:
fig,ax=pyplot.subplots(3,3,sharex=True, sharey=True, figsize=(4,4),dpi=100)
for i in range(3):
  for j in range(3):
    ax[i,j].imshow(train_x[i*3+j])
pyplot.show()

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

In [ ]:
(x_train,y_train), (x_test,y_test)=mnist.load_data()

In [ ]:
x_train.shape

In [ ]:
x_train=x_train.reshape((x_train.shape[0],28,28,1))
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))

In [ ]:
x_train

In [ ]:
x_train=x_train.astype('float32')
x_test=x_test.astype('float32')

In [ ]:
x_test

In [ ]:
datagen=ImageDataGenerator(rotation_range=90)

In [ ]:
datagen.fit(x_train)

In [ ]:
x_batch , y_batch = datagen.flow(x_train, y_train, batch_size=9,shuffle=False,save_to_dir ='/content/drive/MyDrive/image/porosity_filtering/filter_number',save_prefix ='mnist', save_format ='jpeg')

In [ ]:
a=datagen.flow(x_train, y_train, batch_size=9,shuffle=False)
print(a)
len(a)
type(a)

In [ ]:
a=[[2,1,4,3],[5,6,7,9]]
for i,j,c,d  in a:
  for b in range(2):
    for c in range(2):
       #d=i[b]+j[c]
       print(b,c)

In [ ]:
a = [[2, 1, 4, 3], [5, 6, 7, 9]]

for i, j in a:
    print('salam')

In [ ]:
a = [[2, 3], [4, 5]]

for i, j in a:
    print(i, j)

In [ ]:
a=[2,1]
for i,j in a:
  d=print(i+j)
  print('salam')

In [ ]:
a=[[2,1]]
for i,j in a:
  d=print(i+j)
  print('salam')

In [ ]:
t,e=a
t

In [ ]:
a=[[2,1,8],[5,6,8]]
for i,j,z in a:
  d=print(i+j+z)
  print('salam')

In [ ]:
a=[[2,1],[5,6]]
for i,j in a:
  d=print(i+j)
  print('salam')

In [ ]:
i,j=a
i
j

In [ ]:
w=(np.array([1,2,3]),np.array([7,8,2]))
for i,j in w:
  print(i)

In [ ]:
w=(np.array(1,2,3),np.array(7,8,2))
for i,j in a:
  print(i)


In [ ]:
for x_batch , y_batch in datagen.flow(x_train, y_train, batch_size=9,shuffle=False,save_to_dir ='/content/drive/MyDrive/image/porosity_filtering/filter_number',save_prefix ='mnist', save_format ='jpeg'):
  fig,ax=plt.subplots(3,3,sharex=True, sharey=True, figsize=(4,4))
  for i in range(3):
    for j in range(3):
      ax[i,j].imshow(x_batch[3*i+j],cmap='gray')
  plt.show()
  break

In [ ]:
i=0
for x_batch , y_batch in datagen.flow(x_train, y_train, batch_size=16,shuffle=False,save_to_dir ='/content/drive/MyDrive/image/porosity_filtering/filter_number',save_prefix ='mnist', save_format ='jpeg'):

   plt.imshow(x_batch[i],cmap='gray')
   plt.show()
   i+=1
   if i==20:
     break

In [ ]:
fig, ax = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(4,4))
for i in range(3):
        for j in range(3):
            ax[i][j].imshow(x_train[i*3+j])

In [ ]:
from tensorflow.keras.utils import array_to_img, img_to_array,load_img

In [ ]:
img = load_img('/content/drive/MyDrive/image/porosity_filtering/filter_number/ShowStdPic.jpg')

In [ ]:
sara=img_to_array(img)
type(sara)

In [ ]:
sara.shape
#sara.reshape(500,500,3)

In [ ]:
sara = sara.reshape((1, ) + sara.shape)

In [ ]:
sara.shape

In [ ]:
i=0
for batch in datagen.flow(sara, batch_size = 5,save_to_dir ='/content/drive/MyDrive/image/porosity_filtering/filter_number',save_prefix ='sara', save_format ='jpeg'):

    i += 1
    if i > 3:
        break

In [ ]:
i=0
for x_batch  in datagen.flow(sara, batch_size=1,shuffle=False):
   plt.imshow(x_batch[i],cmap='gray')
   plt.show()
   i+=1
   if i==5:
      break

In [ ]:
import numpy as np
y=np.array([[2,8],[9,3]])
y

In [ ]:
y.shape

In [ ]:
y.reshape(-1,4)

In [ ]:
y.reshape(1,4)

In [ ]:
x=[]
d=np.array([[[4,2],[8,6]],[[9,8],[2,1]]])
d.shape
x.append(d)

In [ ]:
x=np.array(x)
x.shape

In [ ]:
import tensorflow as tf
print(tf.__version__)
